# ⚙️ Módulos y Librerías: Eliminación de Directorios Vacíos (`os.rmdir`)

| Propiedad | Detalle |
| :--- | :--- |
| 📍 **Módulo** | Módulos, Paquetes y Librerías / Módulo OS |
| 🎯 **Objetivo** | Comprender la eliminación de directorios vacíos mediante `os.rmdir()`, identificando sus restricciones de seguridad frente a carpetas con contenido y gestionando excepciones como `OSError` / `DirectoryNotEmptyError` y `FileNotFoundError`. |
| 🛠️ **Conceptos** | `os.rmdir()`, `Eliminación de Directorios Vacíos`, `OSError`, `FileNotFoundError`, `NotADirectoryError` |

---

### 💡 El Método `os.rmdir()`

La función **`os.rmdir(path)`** (*Remove Directory*) borra un directorio en la ruta especificada. Por diseño y seguridad, **solo permite eliminar carpetas que estén completamente vacías**.

---

### 🗂️ Restricciones y Excepciones Comunes

```text
[ Carpeta Vacía: './temp_dir' ] ──( os.rmdir )──> ✅ Directorio Eliminado
[ Carpeta con Archivos: './data_dir' ] ──( os.rmdir )──> ❌ OSError (Directory not empty)
```

| Situación | Comportamiento / Error | Solución Recomendada |
| :--- | :--- | :--- |
| 📁 Directorio con contenido | Lanza `OSError` (o `DirectoryNotEmptyError`). | Vaciar el directorio primero o usar `shutil.rmtree()`. |
| 📄 La ruta es un archivo | Lanza `NotADirectoryError` (o `OSError` en Windows). | Utilizar `os.remove()` para archivos. |
| 🔍 El directorio no existe | Lanza `FileNotFoundError`. | Validar existencia previa con `os.path.exists()`. |
```

In [ ]:
import os
from typing import Any

# 🛰️ [1] Banner visual de inicio de ejecución
print("=" * 80)
print("🐍 MÓDULO OS | Eliminación de Directorios Vacíos con os.rmdir()")
print("=" * 80)

# ================================================================================
# 🧪 1. ELIMINACIÓN BÁSICA DE UN DIRECTORIO VACÍO CON VERIFICACIÓN
# ================================================================================
# Creación de una carpeta temporal vacía y borrado seguro con os.rmdir().
# ================================================================================

print("\n--- 🚀 [PASO 1]: Creación y Borrado Controlado de una Carpeta Vacía ---")

directory_to_remove: str = "directorio_a_eliminar"

# ⚙️ [1] Crear una carpeta de prueba
if not os.path.exists(directory_to_remove):
    os.mkdir(directory_to_remove)

print(f"📋 Estado previo: ¿Existe '{directory_to_remove}'?: {os.path.exists(directory_to_remove)}")

try:
    # ⚙️ [2] Validar que existe y es un directorio antes de eliminar
    if os.path.isdir(directory_to_remove):
        # 🔌 [1] Eliminar el directorio especificado
        os.rmdir(directory_to_remove)
        print(f"  ✅ Directorio '{directory_to_remove}' eliminado exitosamente con os.rmdir().")
    else:
        print(f"  ⚠️ El elemento '{directory_to_remove}' no existe o no es un directorio.")

finally:
    # 🔌 [2] Confirmar la eliminación del directorio
    print(f"📋 Estado posterior: ¿Existe '{directory_to_remove}'?: {os.path.exists(directory_to_remove)}")

## 🧱 Control de Excepciones Comunes en `os.rmdir()`

### 💡 Captura de OSError (Directory Not Empty) y FileNotFoundError:
Intentar eliminar una carpeta que contiene archivos o carpetas hijas con `os.rmdir()` provocará un error explícito del sistema de archivos.

In [ ]:
# ================================================================================
# 🧪 2. MANEJO DE ERRORES AL INTENTAR ELIMINAR DIRECTORIOS CON CONTENIDO
# ================================================================================
print("\n--- 🚀 [PASO 2]: Captura de Errores (OSError y FileNotFoundError) ---")

non_empty_folder: str = "carpeta_con_contenido_demo"
non_existent_folder: str = "carpeta_inexistente_demo"

# Crear carpeta con un archivo dentro para forzar la excepción
os.makedirs(non_empty_folder, exist_ok=True)
dummy_file_path: str = os.path.join(non_empty_folder, "archivo_interno.txt")
with open(dummy_file_path, "w", encoding="utf-8") as dummy_file:
    dummy_file.write("Contenido que impide el uso de rmdir.")

try:
    # ⚙️ [1] Intento de eliminar una carpeta que NO está vacía
    print(f"📋 Intento 1: Borrando carpeta no vacía '{non_empty_folder}'...")
    os.rmdir(non_empty_folder)

except OSError as error:
    print("  ❌ Excepción capturada: No se puede eliminar una carpeta con contenido.")
    print(f"  🔍 Detalle técnico: {error}")

try:
    # ⚙️ [2] Intento de eliminar una carpeta inexistente
    print(f"\n📋 Intento 2: Borrando carpeta inexistente '{non_existent_folder}'...")
    os.rmdir(non_existent_folder)

except FileNotFoundError as error:
    print("  ❌ Excepción capturada: La carpeta especificada no existe.")
    print(f"  🔍 Detalle técnico: {error}")

finally:
    # Limpieza del contenido interno y de la carpeta de prueba
    if os.path.exists(dummy_file_path):
        os.remove(dummy_file_path)
    if os.path.exists(non_empty_folder):
        os.rmdir(non_empty_folder)
        print("  🔒 Entorno de prueba limpiado correctamente.")

## 🛒 Ejercicios Prácticos Avanzados Integrados

---

### 🧪 Ejercicio 1: Limpiador de Estructuras de Carpetas Huérfanas

**Escenario Empresarial:** Se necesita una función de mantenimiento que revise una lista de rutas de carpetas de almacenamiento temporal y elimine únicamente aquellas que hayan quedado vacías, omitiendo de forma segura las carpetas que sigan conteniendo información.

In [ ]:
# ================================================================================
# 🧪 3. EJERCICIO PRÁCTICO 1: DESTRUCTOR DE DIRECTORIOS HUÉRFANOS Y VACÍOS
# ================================================================================
print("\n--- 🧪 [PASO 3.1]: Ejercicio 1 - Limpiador Idempotente de Carpetas Vacías ---")


def purge_empty_directories(folder_paths: list[str]) -> dict[str, Any]:
    """Inspecciona y elimina únicamente los directorios de la lista que estén vacíos.

    Args:
        folder_paths (list[str]): Lista de rutas de directorios a evaluar.

    Returns:
        dict[str, Any]: Reporte detallado de carpetas borradas, omitidas y fallidas.
    """
    report: dict[str, Any] = {
        "deleted": [],
        "skipped_not_empty": [],
        "not_found": [],
        "failed": []
    }

    for path in folder_paths:
        try:
            # [1] Verificar existencia y que sea directorio
            if not os.path.exists(path):
                report["not_found"].append(path)
                print(f"  ⚠️ Omitido: '{path}' (No existe)")
                continue

            if not os.path.isdir(path):
                report["failed"].append({"path": path, "reason": "No es un directorio"})
                print(f"  ❌ Falló: '{path}' (Es un archivo)")
                continue

            # [2] Verificar si está vacío usando os.listdir()
            if len(os.listdir(path)) > 0:
                report["skipped_not_empty"].append(path)
                print(f"  ⚠️ Omitido: '{path}' (Contiene elementos)")
                continue

            # [3] Proceder a la eliminación
            os.rmdir(path)
            report["deleted"].append(path)
            print(f"  ✅ Directorio vacío eliminado: '{path}'")

        except PermissionError:
            report["failed"].append({"path": path, "reason": "Permisos insuficientes"})
            print(f"  ❌ Falló: '{path}' (Sin permisos)")

        except OSError as error:
            report["failed"].append({"path": path, "reason": str(error)})
            print(f"  ❌ Falló: '{path}' ({error})")

    return report


# Crear escenarios de prueba
empty_dir_1: str = "temp_empty_alpha"
empty_dir_2: str = "temp_empty_beta"
occupied_dir: str = "temp_occupied_gamma"

os.makedirs(empty_dir_1, exist_ok=True)
os.makedirs(empty_dir_2, exist_ok=True)
os.makedirs(occupied_dir, exist_ok=True)

# Agregar un archivo en occupied_dir
with open(os.path.join(occupied_dir, "keep_me.tmp"), "w", encoding="utf-8") as f:
    f.write("Keep this file.")

candidates: list[str] = [empty_dir_1, empty_dir_2, occupied_dir, "missing_dir_delta"]

print("📋 Ejecutando purga de carpetas vacías...")
purge_results: dict[str, Any] = purge_empty_directories(candidates)

print("\n📋 Resumen del Informe de Purga:")
print(f"  🔹 Carpetas eliminadas ({len(purge_results['deleted'])}): {purge_results['deleted']}")
print(f"  🔹 Conservadas por contenido ({len(purge_results['skipped_not_empty'])}): {purge_results['skipped_not_empty']}")
print(f"  🔹 Rutas no encontradas ({len(purge_results['not_found'])}): {purge_results['not_found']}")

# Limpieza final de la carpeta conservada en las pruebas
if os.path.exists(occupied_dir):
    os.remove(os.path.join(occupied_dir, "keep_me.tmp"))
    os.rmdir(occupied_dir)

# --------------------------------------------------------------------------------
# 🏁 [PASO 4]: Cierre de Ejecución
# --------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("✅ [FIN DE EJECUCIÓN] Lección de os.rmdir() finalizada.")
print("=" * 80)